<img src="images/corndel_logo.png" height="200">

# Advanced Python & pandas for data engineering


## Key

> 💻 Activity for you to do <br>

>❓ A question for you <br>

> ℹ️ Important note (e.g. data security reminder) <br>

## Background

In this notebook, we build upon the foundational concepts introduced in the previous "Core Python & pandas for data engineering" notebook. We'll explore more advanced data engineering techniques using the same NHS Accident & Emergency dataset.

### Advanced NHS Data Engineering Task

As a senior data engineer in the NHS, you're now tasked with enhancing the data pipeline to support more complex analytics and prepare for future scaling. The data analyst you're collaborating with has requested the following improvements:

1. **Modular Pipeline**: Develop a more flexible and maintainable pipeline structure using separate functions for extract, transform, and load operations.
2. **Enhanced Data Quality**: Implement basic error handling and data validation checks.
3. **Performance Optimisation**: Compare CSV and Parquet file formats for efficient data storage and retrieval.
4. **Testing**: Implement basic unit tests for key transformation functions to ensure data integrity.
5. **Logging**: Add logging throughout the pipeline to track execution and aid in debugging.

These enhancements will enable more sophisticated analytics, improve data reliability, and prepare the pipeline for future scaling in a cloud environment. They will also promote better code organisation and maintainability.

In this notebook, we will:
- Set up a virtual environment for our Python packages
- Create modular functions for data extraction, transformation, and loading
- Implement basic error handling and data validation
- Compare CSV and Parquet file formats
- Create simple unit tests for our transformation functions
- Add logging to our data pipeline

We'll be working within a Jupyter Notebook environment, which offers an interactive and visually clear format ideal for teaching and learning. However, at the end of the exercises we'll also discuss how this notebook-based approach could be transitioned into a more production-ready Python script structure in the future.

> **ℹ️** Remember that we're still working with the artificial NHS A&E dataset. In a real-world scenario, you would need to consider additional security and compliance measures when handling sensitive healthcare data.

---

## Create and use Virtual Python Environment

Although this notebook runs in JupyterLite, which already has the packages you need, when you work on your own machine it is beneficial to create and use a virtual environment (`venv`). A virtual environment helps keep your project dependencies separate from other projects and the system-wide packages. This practice avoids conflicts and ensures consistency. Additionally, using virtual environments allows you to specify the exact versions of the packages you need, which is important for making sure your project works the same way on different machines and when sharing your code with others. For example, review the release notes for each version of Pandas and note how each release brings new functionality and old functionality can be deprecated and break our code: https://pandas.pydata.org/docs/whatsnew/

> ℹ️ JupyterLite runs in your browser and has no terminal, so you cannot create a virtual environment here: packages are installed with `%pip install` cells instead. Read the steps below to see how you would do this in VS Code on your own machine, then carry on with the notebook.

First, open the integrated terminal in VS Code.

1. Create a virtual environment named 'venv' https://docs.python.org/3/tutorial/venv.html#creating-virtual-environments
    - `python -m venv venv`
2. Activate the virtual environment:
    - `source venv/bin/activate`
3. With the virtual environment activated, install the necessary packages using pip.
    - `pip install notebook==7.0.7 pandas==2.0.3 pyarrow==15.0.0`
4. Once the previous command has completed, select this virtual environment by using the "kernel picker" as shown in this VS Code guidance: https://code.visualstudio.com/docs/datascience/jupyter-notebooks#_create-or-open-a-jupyter-notebook

![kernel-picker](images/native-kernel-picker.png)

*Image from the Visual Studio Code documentation, https://code.visualstudio.com/docs/datascience/jupyter-notebooks (CC BY 3.0 US)*

--- 

## Extract

Let's begin by setting up our environment and importing the necessary libraries.

In [ ]:
import pandas as pd

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_rows', None)

Then define the extract function for our pipeline:


In [ ]:
def extract_data(file_path):
    df = pd.read_csv(file_path)
    return df

And then extract the data into a dataframe. The CSV file is in this lab, next to the notebook.

In [ ]:
# Extract the NHS A&E data
# (originally read from https://raw.githubusercontent.com/Corndel/corndel-datasets/main/nhs/artificial_hes_ae_2122.csv)
file_path = 'artificial_hes_ae_2122.csv'

df = extract_data(file_path)
df.head()

💻 Read the pandas documentation for how bad lines are dealt with https://pandas.pydata.org/pandas-docs/stable/user_guide/io.html#error-handling While this data is clean with no bad lines, csv files are notoriously prone to having bad lines! Update the `extract_data()` function in the code cell below for the argument `on_bad_lines`, set its value to `warn`.

In [ ]:
# Update function extract_data() function to set `on_bad_lines` value to be `warn`



❓ Within the pandas function `read_csv()`, what is the default value of the argument `on_bad_lines`? https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html 

❓ Can you guess the main reason for the default value of the argument `on_bad_lines`? What problems could setting the value to be `warn` create?

Let's now create a function that carries out any data cleaning. In the funtion below we simply remove any leading or trailing white spaces from columns names.

In [ ]:
def clean_data(df):
    """
    Perform initial data cleaning
    """
    # Remove any leading/trailing whitespace from column names
    df.columns = df.columns.str.strip()
      
    return df

In [ ]:
df = clean_data(df)

💻 There could be other issues in the data that require cleaning. In the code cells below, run the following commands, `df.info(max_cols=df.shape[1])`, then `df.describe()` in the cell below it.
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html

In [ ]:
# Describe dataframe info



In [ ]:
# Describe dataframe column characteristics



❓ Is there any information in the output of the previous two code cells that identifies anything unusual about the data that would merit further investigation and adding an additional cleaning function? For example, 

1. from `.info()` is there an unusually low number of `non-null` values in certain columns? 
2. Is there a column `Dtype` that is not appropriate? This could indicate when parsing the columns and selecting the column `Dtype`, `read_csv()` detected mixed data types (e.g. text in a numeric column).
3. Does `.describe()` reveal any unusual `min` or `max` value or do the percentiles indicate a distribution of data you would not expect? (Note we we explored distributions on this data in our introduction to Python notebook visually using `sweetviz`).

--- 

## Transform

Let's now take the transform logic from our introduction to Python notebook and put each transformatation into its own function.

Note that we put each transform into its own function. We will see why seperate fucntions is better that one single mixed tranform function soon when when introduce testing.

Run the cell below to create the different transformation functions. 

In [ ]:
import pandas as pd
import numpy as np

def convert_arrival_date(df):
    """Convert ARRIVALDATE to datetime."""
    df['ARRIVALDATE'] = pd.to_datetime(df['ARRIVALDATE'], format='%Y-%m-%d')
    return df

def create_arrival_datetime(df):
    """Create ARRIVAL_DATETIME column."""
    df['ARRIVAL_DATETIME'] = pd.to_datetime(
        df['ARRIVALDATE'].astype(str) + ' ' +
        df['ARRIVALTIME'].astype(str).str.zfill(4),
        format='%Y-%m-%d %H%M'
    )
    return df

def extract_date_components(df):
    """Extract date components from ARRIVAL_DATETIME."""
    df['YEAR'] = df['ARRIVAL_DATETIME'].dt.year
    df['MONTH'] = df['ARRIVAL_DATETIME'].dt.month
    df['DAY_OF_WEEK'] = df['ARRIVAL_DATETIME'].dt.day_name()
    df['HOUR'] = df['ARRIVAL_DATETIME'].dt.hour
    df['MINUTE'] = df['ARRIVAL_DATETIME'].dt.minute
    return df

def categorize_duration(duration):
    """Categorize a duration into Short, Medium, Long, or Very Long."""
    if duration <= 60:
        return 'Short'
    elif duration <= 120:
        return 'Medium'
    elif duration <= 240:
        return 'Long'
    else:
        return 'Very Long'

def categorise_durations(df, duration_columns):
    """
    Categorise duration columns into Short, Medium, Long, or Very Long categories.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The input DataFrame containing duration columns
    duration_columns : list
        List of column names representing time durations in minutes 
        (e.g. ['CONCLDUR', 'DEPDUR', 'INITDUR', 'TRETDUR'])
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with new categorical columns added (original column name + '_CATEGORY')
    """
    for col in duration_columns:
        df[f'{col}_CATEGORY'] = df[col].apply(categorize_duration)
    return df

Note that the `categorise_durations` function in the code above uses [Python f-strings](https://www.w3schools.com/python/python_string_formatting.asp) to dynamically create new column names within the DataFrame. F-strings, introduced in Python 3.6, are a more readable and efficient method for inserting variable values directly into strings. They allow us to embed expressions inside strings using curly braces `{}`, making the code concise and less error-prone compared to older methods such as the `%` operator.

In the `categorise_durations` function, f-strings are used to add `_CATEGORY` to each duration column name that is passed into the for loop from the list of duration column names (`duration_columns`). So automatically creating new column names from all the original duration column names but with each one ending in `_CATEGORY`. This approach is better and more flexible than hard coding the duration column names because we only need to update the list of duration columns once, making the code easier to maintain and adapt to changes (e.g. if new duration columns were added or if their name changed).

In the context of our NHS A&E data, this means we can process all four time-based measurements (CONCLDUR, DEPDUR, INITDUR, TRETDUR) with the same single categorisation logic, automatically creating new columns like `CONCLDUR_CATEGORY`, `DEPDUR_CATEGORY`, etc.

> 💻 Now it's your turn to convert one of the previous transformations from the introduction to Python notebook into its own function. Put the following into a function that begins with:  `def create_distance_scenarios(df):` then as the following in the middle...

- `df['SITEDIST_Closer'] = df['SITEDIST'] * 0.75`
- `df['SITEDIST_Farther'] = df['SITEDIST'] * 1.25`
- `df['SITEDIST_MILES'] = df['SITEDIST'] * 0.621371`

...finally, what should the function `return`? Look at the functions above for the answer.

In [ ]:
# Build the create_distance_scenarios() function here



We can now build a transform data function that has all the seperate modular transformations combined

In [ ]:
def transform_data(df, duration_columns):
    """
    Apply all transformations to the NHS A&E data.
    """
    df = convert_arrival_date(df)
    df = create_arrival_datetime(df)
    df = extract_date_components(df)
    df = create_distance_scenarios(df)
    df = categorise_durations(df, duration_columns)
    return df

And then run those functions on our data

In [ ]:
# Run the transform_data() function

# Define the duration columns
duration_columns = ['CONCLDUR', 'DEPDUR', 'INITDUR', 'TRETDUR']

df = transform_data(df, duration_columns)
df.head()

--- 

## Introduction to Testing

Before we move on to loading the data, let's briefly introduce and experiment with testing.

We will cover testing in much more detail later in the programme using [pytest](https://docs.pytest.org/en/8.2.x/index.html) and [Great Expectations](https://greatexpectations.io/).

As an introduction to testing, below we will create a simple test together.

Think of testing as exactly what you might do manually to test that some of your logic works, except you formalise your manual test into code so that it can be executed automatically, ensuring consistency and saving time whenever you need to verify the correctness of your code.

### Why is Testing Important?

We should aim to test key functions that transform data in data engineering, focusing on areas where errors would have significant impact or where changes are likely, in order to make our pipelines more robust to both changes in the data and changes to our code, without over-testing or creating unnecessary maintenance overhead.

For example, consider the function you just created that categorises durations, `categorise_durations`. This function might look straightforward, but ensuring it behaves correctly for all possible inputs is a sensible proactive approach. For instance, what if a duration is exactly 60 minutes? Or what if it's a very large number? Such scenarios could lead to incorrect categorisation in our analysis.

By adding simple tests, we can catch potential issues early. The `assert` statement in Python is used to check if a given condition is true. If the condition is false, it raises an `AssertionError` with an optional error message. In testing, `assert` statements verify the expected behaviour of code.

For example, consider this line: `assert df['CONCLDUR_CATEGORY'].iloc[0] == 'Short'`

This checks if the first value in the `CONCLDUR_CATEGORY` column is 'Short'. If not, an `AssertionError` is raised, indicating a failure in the test. Using `assert` statements helps ensure that the code behaves as expected and allows for easy identification of issues when assertions fail. This helps pinpoint the exact location and nature of the issue, making it easier to debug and fix.

### Additional Benefits of Testing

1. **Detection of Issues from Code Changes**: Tests can catch problems introduced by future changes to the code, ensuring that new updates do not break existing functionality.
2. **Documentation of Logic**: Well-written tests serve as documentation for the expected behaviour of code, helping new developers understand the logic without extensive reading.
3. **Facilitates Refactoring**: With a comprehensive test suite, you can refactor code confidently, knowing that tests will catch any deviations from expected behaviour.
4. **Improves Code Quality**: Writing tests encourages better code practices and design, as the need to test the code can lead to more modular and maintainable code structures.

### Creating a Simple Test

For now, run the cell below to create some simple test data and run a basic test using Python's `assert` statement. You can learn more about the `assert` statement in [this Real Python guide](https://realpython.com/python-assert-statement/).


In [ ]:
# Create test data
def create_test_data():
    return pd.DataFrame({
        'CONCLDUR': [45, 180, 60, 241],
        'DEPDUR': [60, 120, 240, 300],
        'INITDUR': [15, 30, 45, 59],
        'TRETDUR': [30, 60, 121, 240]
    })

# Create test data
test_df = create_test_data()
test_df.head()

We now apply our transformation functions to that test data and view the resuts.

In [ ]:
# Define the duration columns
duration_columns = ['CONCLDUR', 'DEPDUR', 'INITDUR', 'TRETDUR']

# Transform test data
transformed_df = categorise_durations(test_df, duration_columns)

# Display the transformed test data
transformed_df.head()

We could of course manually inspect the results above to see if the transform is correct, however, because our test data is fixed in code above, we know exactly what the result should always be and can `assert` that is the case in the tests below.

In [ ]:
def test_categorise_durations():
    df = create_test_data()
    duration_columns = ['CONCLDUR', 'DEPDUR', 'INITDUR', 'TRETDUR']
    df = categorise_durations(df, duration_columns)
    
    # Test CONCLDUR
    assert df['CONCLDUR_CATEGORY'].iloc[0] == 'Short', "45 minutes should be categorised as Short"
    assert df['CONCLDUR_CATEGORY'].iloc[1] == 'Long', "180 minutes should be categorised as Long"
    assert df['CONCLDUR_CATEGORY'].iloc[2] == 'Short', "60 minutes should be categorised as Short"
    assert df['CONCLDUR_CATEGORY'].iloc[3] == 'Very Long', "241 minutes should be categorised as Very Long"
    
    # Test DEPDUR
    assert df['DEPDUR_CATEGORY'].iloc[0] == 'Short', "60 minutes should be categorised as Short"
    assert df['DEPDUR_CATEGORY'].iloc[1] == 'Medium', "120 minutes should be categorised as Medium"
    assert df['DEPDUR_CATEGORY'].iloc[2] == 'Long', "240 minutes should be categorised as Long"
    assert df['DEPDUR_CATEGORY'].iloc[3] == 'Very Long', "300 minutes should be categorised as Very Long"
    
    # Test INITDUR
    assert all(df['INITDUR_CATEGORY'] == 'Short'), "All INITDUR values should be categorised as Short"
    
    # Test TRETDUR 
    # Add assertions here
    
    print("test_categorise_durations passed.")

# Run the test
test_categorise_durations()

> 💻 You should find that the test for the `categorise_durations` function passes. However, it is missing some final asserts for you to add.

1. Review the `categorise_durations` function to understand what it is supposed to do.
2. For the `test_categorise_durations` function:
   - Consider the expected categories for the `TRETDUR` column based on the given durations. What categories should the values in this column be classified as? Short, Medium or Long?
   - Add assertions to check that the values in `TRETDUR_CATEGORY` are correctly categorised.

--- 

## Load Data to a Parquet File

In this final step of our data pipeline, we will save the transformed data to a [Parquet](https://parquet.apache.org/docs/overview/) file. Parquet is a columnar storage file format that is highly efficient for both storage and retrieval, especially for large datasets. It is particularly well-suited for data engineering tasks due to its compression and query performance benefits.

Below we define a function `load_data_to_parquet` that takes a DataFrame and a file path as arguments. This function will save the DataFrame to the specified file path in Parquet format.

With all our transformation functions defined, we can now run the complete ETL pipeline:

- Extract the Data: We extract the NHS A&E data from the provided CSV file.
- Clean the Data: We clean the data by stripping any leading or trailing whitespace from the column names.
- Transform the Data: We apply all the transformation functions to process the data as required.
- Load the Transformed Data: Finally, we load the transformed data to a Parquet file for efficient storage and retrieval.


Writing Parquet files needs a Parquet engine. In the browser, pandas uses `fastparquet`, so install it first.

In [ ]:
%pip install -q fastparquet

In [ ]:
def load_data_to_parquet(df, file_path):
    """Save DataFrame to a Parquet file."""
    df.to_parquet(file_path, index=False)

# Load data
df = extract_data(file_path)

# Clean data
df = clean_data(df)

# Define duration columns
duration_columns = ['CONCLDUR', 'DEPDUR', 'INITDUR', 'TRETDUR']

# Transform data
df = transform_data(df, duration_columns)

# Save the transformed data to a parquet file
output_file_path = 'transformed_nhs_ae_data.parquet'
load_data_to_parquet(df, output_file_path)


 > 💻 Run the provided code below to save the transformed DataFrame to both CSV and Parquet files. This will help you understand the benefits of using Parquet over CSV by comparing the file sizes, read/write performance, and how reliably each format stores data types. Observe the printed results, reflecting on the file sizes and read times for both formats.

In [ ]:
import os
import time
def load_data_to_csv(df, file_path):
    """Save DataFrame to a CSV file."""
    df.to_csv(file_path, index=False)

# Save the transformed data to a CSV file
output_file_path_csv = 'transformed_nhs_ae_data.csv'
load_data_to_csv(df, output_file_path_csv)

# Save the transformed data to a Parquet file
output_file_path_parquet = 'transformed_nhs_ae_data.parquet'
load_data_to_parquet(df, output_file_path_parquet)

# Measure and compare file sizes
csv_file_size = os.path.getsize(output_file_path_csv)
parquet_file_size = os.path.getsize(output_file_path_parquet)
print(f"CSV file size: {csv_file_size} bytes")
print(f"Parquet file size: {parquet_file_size} bytes")

# Measure and compare read times
start_time = time.time()
df_csv = pd.read_csv(output_file_path_csv)
csv_read_time = time.time() - start_time
print(f"CSV read time: {csv_read_time} seconds")

start_time = time.time()
df_parquet = pd.read_parquet(output_file_path_parquet)
parquet_read_time = time.time() - start_time
print(f"Parquet read time: {parquet_read_time} seconds")

> ❓ What are the file sizes of the CSV and Parquet files?

> ❓ Review the `.to_parquet()` pandas documentation, which default compression algorithm is used? [Pandas Documentation](https://pandas.pydata.org/pandas-docs/version/1.1/reference/api/pandas.DataFrame.to_parquet.html)

> ❓ How do the read times for the CSV and Parquet files compare?

> ❓ Data types can only be inferred from a CSV but are stored as metadata in a Parquet file. How could this be advantageous in ensuring data consistency and preventing data type errors in your data pipeline? How can you inspect the data types stored in a Parquet file? (Hint: Use `pandas.read_parquet()` and check the DataFrame's `dtypes` attribute)

> ❓ A CSV file is human-readable and can be easily opened with a text editor, whereas a Parquet file requires specific tools to read and inspect. What are some potential downsides of using Parquet files, and how might the need for special tools impact your workflow or the accessibility of your data?

--- 

## Adding Logging to a Simple Data Pipeline

In this section, we'll explore how to add basic logging to a simplified data pipeline. Logging is a crucial tool for tracking the progress of your data processing tasks, identifying issues, and facilitating debugging. Even in a straightforward pipeline like this one, logging can provide valuable insights into the execution flow that include:

- **Track Progress**: Understand which stage of the pipeline is currently executing.
- **Identify Issues**: Quickly spot any errors or unexpected behavior.
- **Debugging**: Get detailed information to help diagnose and fix problems.

We'll use Python's built-in `logging` module, which is simple to set up and use. Our pipeline will demonstrate logging in three main stages:

1. **Extract**: Loading data from a CSV file.
2. **Transform**: Applying basic transformations to the data.
3. **Load**: Saving the processed data to a new CSV file.

We start by configuring basic logging at the beginning of our script:

In [ ]:
import pandas as pd
import logging
from datetime import datetime

# Set up basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# File path for the data
# (originally read from https://raw.githubusercontent.com/Corndel/corndel-datasets/main/nhs/artificial_hes_ae_2122.csv)
file_path = 'artificial_hes_ae_2122.csv'

def extract_data(file_path):
    """Extract the NHS A&E data from CSV."""
    logging.info("Extracting data from CSV file")
    return pd.read_csv(file_path)

def transform_data(df):
    """Apply transformations to the data."""
    logging.info("Transforming data")
    
    # Clean column names
    df.columns = df.columns.str.strip()
    
    # Convert ARRIVALDATE to datetime
    df['ARRIVALDATE'] = pd.to_datetime(df['ARRIVALDATE'])
    
    # Create ARRIVAL_DATETIME column
    df['ARRIVAL_DATETIME'] = pd.to_datetime(df['ARRIVALDATE'].astype(str) + ' ' + 
                                            df['ARRIVALTIME'].astype(str).str.zfill(4),
                                            format='%Y-%m-%d %H%M')
    
    # Extract date components
    df['YEAR'] = df['ARRIVAL_DATETIME'].dt.year
    df['MONTH'] = df['ARRIVAL_DATETIME'].dt.month
    df['DAY_OF_WEEK'] = df['ARRIVAL_DATETIME'].dt.day_name()
    
    # Create SITEDIST in miles
    df['SITEDIST_MILES'] = df['SITEDIST'] * 0.621371
    
    return df

def load_data(df, output_file):
    """Load the transformed data to a CSV file."""
    logging.info(f"Loading data to {output_file}")
    df.to_csv(output_file, index=False)

# Run the ETL pipeline
logging.info("Starting the NHS A&E data ETL pipeline")

# Extract
df = extract_data(file_path)

# Transform
df = transform_data(df)

# Load
output_file = f'transformed_nhs_ae_data_{datetime.now().strftime("%Y%m%d")}.csv'
load_data(df, output_file)

logging.info("NHS A&E data ETL pipeline completed successfully")

# Display the first few rows of the transformed data
df.head()


In our enhanced data pipeline below, we've added several extra logging messages that provide insights into our data. Let's examine these additions and then customize them further.

1. In the `extract_data` function:
   - Logs the number of rows and columns in the extracted dataset.

2. In the `transform_data` function:
   - Logs the number of unique values in the 'AEDEPTTYPE' column before and after transformation.
   - Logs the date range (earliest and latest dates) in the 'ARRIVALDATE' column.

3. In the `load_data` function:
   - Logs the total file size of the saved CSV in megabytes.

>  💻 Modify the existing logging messages below (or add new ones) to gain different insights from the data. Here are some ideas:

1. In the `extract_data` function:
   - Log the names of the first 5 columns in the dataset.
     - Hint: Use `df.columns[:5].tolist()`
   - Log the number of non-null values in a specific column of your choice.
     - Hint: Use `df['column_name'].notnull().sum()`

2. In the `transform_data` function:
   - Instead of AEDEPTTYPE, log the unique values for a different categorical column.
     - Hint: Use `df['column_name'].unique()`
   - Log the mean and median of a numerical column (e.g., 'DEPDUR' or 'TRETDUR').
     - Hint: Use `df['column_name'].mean()` and `df['column_name'].median()`
   - Log the most common day of the week for A&E visits.
     - Hint: Use `df['DAY_OF_WEEK'].mode().iloc[0]`

3. In the `load_data` function:
   - Log the number of columns that have been added during the transformation process.
     - Hint: Compare `len(df.columns)` with the original number of columns

In [ ]:
import pandas as pd
import logging
from datetime import datetime
import os

# Set up basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# File path for the data
# (originally read from https://raw.githubusercontent.com/Corndel/corndel-datasets/main/nhs/artificial_hes_ae_2122.csv)
file_path = 'artificial_hes_ae_2122.csv'

def extract_data(file_path):
    """Extract the NHS A&E data from CSV."""
    logging.info("Extracting data from CSV file")
    df = pd.read_csv(file_path)
    logging.info(f"Extracted {len(df)} rows and {len(df.columns)} columns")
    return df

def transform_data(df):
    """Apply transformations to the data."""
    logging.info("Transforming data")
    
    # Log unique values in AEDEPTTYPE before transformation
    logging.info(f"Unique values in AEDEPTTYPE before transformation: {df['AEDEPTTYPE'].nunique()}")
    
    # Clean column names
    df.columns = df.columns.str.strip()
    
    # Convert ARRIVALDATE to datetime
    df['ARRIVALDATE'] = pd.to_datetime(df['ARRIVALDATE'])
    
    # Log date range
    logging.info(f"Date range: from {df['ARRIVALDATE'].min()} to {df['ARRIVALDATE'].max()}")
    
    # Create ARRIVAL_DATETIME column
    df['ARRIVAL_DATETIME'] = pd.to_datetime(df['ARRIVALDATE'].astype(str) + ' ' + 
                                            df['ARRIVALTIME'].astype(str).str.zfill(4),
                                            format='%Y-%m-%d %H%M')
    
    # Extract date components
    df['YEAR'] = df['ARRIVAL_DATETIME'].dt.year
    df['MONTH'] = df['ARRIVAL_DATETIME'].dt.month
    df['DAY_OF_WEEK'] = df['ARRIVAL_DATETIME'].dt.day_name()
    
    # Create SITEDIST in miles
    df['SITEDIST_MILES'] = df['SITEDIST'] * 0.621371
    
    # Log unique values in AEDEPTTYPE after transformation
    logging.info(f"Unique values in AEDEPTTYPE after transformation: {df['AEDEPTTYPE'].nunique()}")
    
    return df

def load_data(df, output_file):
    """Load the transformed data to a CSV file."""
    logging.info(f"Loading data to {output_file}")
    df.to_csv(output_file, index=False)
    file_size = os.path.getsize(output_file) / (1024 * 1024)  # Convert to MB
    logging.info(f"File saved. Total size: {file_size:.2f} MB")

# Run the ETL pipeline
logging.info("Starting the NHS A&E data ETL pipeline")

# Extract
df = extract_data(file_path)

# Transform
df = transform_data(df)

# Load
output_file = f'transformed_nhs_ae_data_{datetime.now().strftime("%Y%m%d")}.csv'
load_data(df, output_file)

logging.info("NHS A&E data ETL pipeline completed successfully")

# Display the first few rows of the transformed data
print(df.head())

>  ❓ How do the new log messages help you understand the data better?

>  ❓ In what scenarios might these additional logs be particularly useful?

>  ❓ Can you think of any other logging additions that would be valuable for this dataset?


---

## Converting Jupyter experiments to production Python code

In this lesson, we have used a Jupyter Notebook to explore and develop our data pipeline. This environment allows running code cells individually for step-by-step exploration and immediate feedback. Jupyter Notebooks display various types of output inline, making it easy to visualise data and results. Interspersing code with explanatory text creates a self-documenting workflow, useful for understanding processes. The ease of experimentation facilitated by notebooks helps when learning new concepts or debugging.

However, Jupyter Notebooks may not be optimal for production-level, automated data pipelines. Refactoring code into standalone Python (.py) files is a logical next step. This transition offers us many benefits in production, such as better version control with systems like Git and improved modularity by separating functions into different files. Scripts can be more easily scheduled and integrated into automated workflows, a key requirement for data engineering tasks.

In our next workshop, we will practice this conversion of notebook code to a Python script that we automate. Until then, while this Microsoft learn page is for how to [convert a **data science** pipeline to production Python code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks#_create-or-open-a-jupyter-notebook), the lessons and techniques here on removing nonessential code, refactoring into functions, and combining related functions into Python files are directly applicable to our data engineering tasks.